# Train B0 — YOLOv8n Baseline (100 epochs) — Kaggle GPU
**KLTN: PCB Defect Detection**

### Cấu trúc dataset Kaggle đã mount:
```
/kaggle/input/datasets/liuxiaolong1/pcb-defect-detection-dataset/
    PKU-Market-PCB(Data enhanced version)/
        train/images/*.jpg  + train/labels/*.txt
        valid/images/*.jpg  + valid/labels/*.txt
        test/images/*.jpg   + test/labels/*.txt

/kaggle/input/datasets/tanthongthai/   ← project code (configs/ + data/splits/)
```

### Cách chạy:
1. Add cả 2 dataset vào notebook
2. Chọn **GPU T4 x2** + **Internet ON**
3. **Run All**


In [18]:
# Fix Ray Tune conflict
!pip uninstall ray -y -q 2>/dev/null || True
!pip install ultralytics==8.3.40 -q --upgrade


In [19]:
import os, json, shutil, subprocess, time, zipfile
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import yaml
from ultralytics import YOLO

# ── Kaggle dataset paths ────────────────────────────────────────────────────
# Dataset ảnh gốc (liuxiaolong1)
KAGGLE_IMG_ROOT = Path(
    "/kaggle/input/datasets/liuxiaolong1/pcb-defect-detection-dataset"
    "/PKU-Market-PCB(Data enhanced version)"
)

# Dataset code project (tanthongthai) — chứa configs/ và data/splits/
KAGGLE_CODE_ROOT = Path("/kaggle/input/datasets/tanthongthai")

WORK_DIR = Path("/kaggle/working")

# ── Verify datasets tồn tại ─────────────────────────────────────────────────
assert KAGGLE_IMG_ROOT.exists(), f"❌ Không tìm thấy dataset ảnh: {KAGGLE_IMG_ROOT}"

print("✅ Dataset ảnh:", KAGGLE_IMG_ROOT)
print("   Train images:", len(list((KAGGLE_IMG_ROOT/"train"/"images").glob("*.jpg"))))
print("   Valid images:", len(list((KAGGLE_IMG_ROOT/"valid"/"images").glob("*.jpg"))))
print("   Test images :", len(list((KAGGLE_IMG_ROOT/"test" /"images").glob("*.jpg"))))
print()

# Tìm thư mục chứa configs/
CODE_ROOT = None
for candidate in [
    KAGGLE_CODE_ROOT,
    *KAGGLE_CODE_ROOT.glob("*/"),
    *KAGGLE_CODE_ROOT.glob("*/*/"),
]:
    if (candidate / "configs" / "base.yaml").exists():
        CODE_ROOT = candidate
        break

if CODE_ROOT:
    print(f"✅ Project code: {CODE_ROOT}")
else:
    print("⚠️  Không tìm thấy configs/base.yaml trong tanthongthai dataset")
    print("   Sẽ dùng hyperparams mặc định.")


✅ Dataset ảnh: /kaggle/input/datasets/liuxiaolong1/pcb-defect-detection-dataset/PKU-Market-PCB(Data enhanced version)
   Train images: 2772
   Valid images: 366
   Test images : 366

✅ Project code: /kaggle/input/datasets/tanthongthai/kltn-pcb-code


In [20]:
# ── Load từ base.yaml nếu có, fallback về default ───────────────────────────
if CODE_ROOT and (CODE_ROOT / "configs" / "base.yaml").exists():
    with open(CODE_ROOT / "configs" / "base.yaml") as f:
        cfg = yaml.safe_load(f)
    print("✅ Loaded configs/base.yaml:", cfg)
else:
    cfg = {"batch": 16, "imgsz": 640, "seed": 41, "patience": 30}
    print("⚠️  Dùng hyperparams mặc định:", cfg)

# ===== CHỈ OVERRIDE EPOCHS — mọi tham số khác từ base.yaml =====
EPOCHS   = 100
BATCH    = cfg.get("batch",    16)
IMGSZ    = cfg.get("imgsz",   640)
SEED     = cfg.get("seed",     41)
PATIENCE = cfg.get("patience", 30)
# ================================================================

print(f"\nHyperparams sẽ dùng:")
print(f"  epochs  : {EPOCHS}  (override)")
print(f"  batch   : {BATCH}")
print(f"  imgsz   : {IMGSZ}")
print(f"  seed    : {SEED}")
print(f"  patience: {PATIENCE}")


✅ Loaded configs/base.yaml: {'model': 'yolov8n.pt', 'imgsz': 640, 'epochs': 20, 'batch': 16, 'optimizer': 'auto', 'seed': 41, 'patience': 30, 'classes': ['missing_hole', 'mouse_bite', 'open_circuit', 'short', 'spur', 'spurious_copper'], 'split_ratio': [0.7, 0.15, 0.15]}

Hyperparams sẽ dùng:
  epochs  : 100  (override)
  batch   : 16
  imgsz   : 640
  seed    : 41
  patience: 30


In [21]:
# ── Load split files từ project code (hoặc dùng thư mục ảnh trực tiếp) ─────

# Prefix cần thay thế trong txt files của project local
LOCAL_PREFIX = "data/raw/PKU-Market-PCB(Data enhanced version)/"

def remap_split(txt_path: Path, kaggle_root: Path) -> list[str]:
    """
    Đọc split .txt, remap relative path → Kaggle absolute path.
    
    Local  : data/raw/PKU-Market-PCB(Data enhanced version)/train/images/xx.jpg
    Kaggle : /kaggle/input/.../PKU-Market-PCB(Data enhanced version)/train/images/xx.jpg
    """
    lines = txt_path.read_text().strip().splitlines()
    remapped = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if line.startswith(LOCAL_PREFIX):
            rel = line[len(LOCAL_PREFIX):]   # e.g. "train/images/xx.jpg"
            new_path = str(kaggle_root / rel)
        else:
            # fallback: thử ghép trực tiếp
            new_path = str(kaggle_root / line)
        remapped.append(new_path)
    return remapped

# ── Thử load từ tanthongthai dataset ─────────────────────────────────────────
splits_loaded = False

if CODE_ROOT:
    train_txt = CODE_ROOT / "data" / "splits" / "train.txt"
    val_txt   = CODE_ROOT / "data" / "splits" / "val.txt"
    
    if train_txt.exists() and val_txt.exists():
        train_paths = remap_split(train_txt, KAGGLE_IMG_ROOT)
        val_paths   = remap_split(val_txt,   KAGGLE_IMG_ROOT)
        splits_loaded = True
        print(f"✅ Loaded splits từ project code: train={len(train_paths)}, val={len(val_paths)}")

# ── Fallback: scan thư mục ảnh trực tiếp ─────────────────────────────────────
if not splits_loaded:
    print("⚠️  Không load được split → scan toàn bộ thư mục ảnh")
    import random
    random.seed(SEED)
    
    all_imgs = sorted((KAGGLE_IMG_ROOT / "train" / "images").glob("*.jpg"))
    all_imgs += sorted((KAGGLE_IMG_ROOT / "valid" / "images").glob("*.jpg"))
    all_imgs = [str(p) for p in all_imgs]
    
    random.shuffle(all_imgs)
    n_val = int(len(all_imgs) * 0.15)
    val_paths   = all_imgs[:n_val]
    train_paths = all_imgs[n_val:]
    splits_loaded = True
    print(f"   Scan fallback: train={len(train_paths)}, val={len(val_paths)}")

# ── Verify sample paths tồn tại ──────────────────────────────────────────────
def check_paths(paths, name, n=5):
    missing = [p for p in paths[:n] if not Path(p).exists()]
    if missing:
        print(f"⚠️  {name}: {len(missing)}/{n} sample paths KHÔNG tồn tại:")
        for p in missing:
            print(f"   {p}")
        return False
    print(f"✅ {name}: {n} sample paths tồn tại")
    return True

ok_train = check_paths(train_paths, "train")
ok_val   = check_paths(val_paths,   "val")

if not (ok_train and ok_val):
    raise RuntimeError("Path remapping thất bại. Kiểm tra lại KAGGLE_IMG_ROOT.")


✅ Loaded splits từ project code: train=3989, val=368
✅ train: 5 sample paths tồn tại
✅ val: 5 sample paths tồn tại


In [22]:
# ── Ghi split files vào working dir ──────────────────────────────────────────
WORK_DIR.mkdir(parents=True, exist_ok=True)
splits_dir = WORK_DIR / "splits"
splits_dir.mkdir(exist_ok=True)

train_txt_kaggle = splits_dir / "train.txt"
val_txt_kaggle   = splits_dir / "val.txt"

train_txt_kaggle.write_text("\n".join(train_paths) + "\n")
val_txt_kaggle.write_text(  "\n".join(val_paths)   + "\n")

print(f"✅ Wrote train.txt: {len(train_paths)} images")
print(f"✅ Wrote val.txt  : {len(val_paths)} images")

# ── Tạo data.yaml cho Kaggle ──────────────────────────────────────────────────
data_yaml = {
    "path"  : str(WORK_DIR),          # base path (Ultralytics dùng để resolve)
    "train" : str(train_txt_kaggle),  # absolute path → Ultralytics dùng trực tiếp
    "val"   : str(val_txt_kaggle),
    # test KHÔNG được thêm để tránh data leakage
    "nc"    : 6,
    "names" : {
        0: "missing_hole",
        1: "mouse_bite",
        2: "open_circuit",
        3: "short",
        4: "spur",
        5: "spurious_copper",
    },
}

data_yaml_path = WORK_DIR / "data_kaggle.yaml"
with open(data_yaml_path, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False, allow_unicode=True)

print(f"\n✅ data_kaggle.yaml:")
with open(data_yaml_path) as f:
    print(f.read())


✅ Wrote train.txt: 3989 images
✅ Wrote val.txt  : 368 images

✅ data_kaggle.yaml:
names:
  0: missing_hole
  1: mouse_bite
  2: open_circuit
  3: short
  4: spur
  5: spurious_copper
nc: 6
path: /kaggle/working
train: /kaggle/working/splits/train.txt
val: /kaggle/working/splits/val.txt



In [24]:
# Fix Ray Tune incompatibility — PHẢI chạy trước model.train()

# Fix 1: patch ray session API bị missing
try:
    import ray.train._internal.session as _ray_session
    if not hasattr(_ray_session, '_get_session'):
        _ray_session._get_session = lambda: None
        print("✅ Patched ray._get_session")
except Exception as e:
    print(f"ray patch skipped: {e}")

# Fix 2: vô hiệu hóa thẳng Ultralytics raytune callback
try:
    import ultralytics.utils.callbacks.raytune as _raytune_cb
    _raytune_cb.on_fit_epoch_end = lambda trainer: None
    print("✅ Disabled ultralytics raytune callback")
except Exception as e:
    print(f"raytune patch skipped: {e}")


✅ Patched ray._get_session
✅ Disabled ultralytics raytune callback


In [25]:
RUN_NAME = "B0_100ep"
t0 = time.time()

model = YOLO("yolov8n.pt")   # pretrained ImageNet weights

results = model.train(
    data     = str(data_yaml_path),
    epochs   = EPOCHS,
    batch    = BATCH,
    imgsz    = IMGSZ,
    seed     = SEED,
    patience = PATIENCE,
    project  = str(WORK_DIR / "runs"),
    name     = RUN_NAME,
    device   = 0,       # GPU Kaggle
    verbose  = True,
    save     = True,
    plots    = True,
    exist_ok = True,
    workers  = 2,
)

elapsed = time.time() - t0
print(f"\n{'='*60}")
print(f"Training hoàn tất! Elapsed: {elapsed/60:.1f} phút")
print(f"{'='*60}")

m = results.results_dict
print(f"mAP50    : {m.get('metrics/mAP50(B)',    0):.4f}")
print(f"mAP50-95 : {m.get('metrics/mAP50-95(B)',0):.4f}")
print(f"Precision: {m.get('metrics/precision(B)',0):.4f}")
print(f"Recall   : {m.get('metrics/recall(B)',   0):.4f}")


New https://pypi.org/project/ultralytics/8.4.161 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.40 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/kaggle/working/data_kaggle.yaml, epochs=100, time=None, patience=30, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=2, project=/kaggle/working/runs, name=B0_100ep, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=41, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, 

train: Scanning /kaggle/input/datasets/liuxiaolong1/pcb-defect-detection-dataset/PKU-Market-PCB(Data enhanced version)/test/labels... 2656 images, 1333 backgrounds, 0 corrupt: 100%|██████████| 3989/3989 [00:03<00:00, 1079.59it/s]

train: WARNING ⚠️ Cache directory /kaggle/input/datasets/liuxiaolong1/pcb-defect-detection-dataset/PKU-Market-PCB(Data enhanced version)/test is not writeable, cache not saved.


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


/usr/local/lib/python3.12/dist-packages/ultralytics/data/augment.py:1850: UserWarning: Argument(s) 'quality_lower' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=75, p=0.0),
val: Scanning /kaggle/input/datasets/liuxiaolong1/pcb-defect-detection-dataset/PKU-Market-PCB(Data enhanced version)/train/labels... 368 images, 0 backgrounds, 0 corrupt: 100%|██████████| 368/368 [00:00<00:00, 776.27it/s]

...

     71/100      2.48G      1.194     0.9108     0.8484         19        640: 100%|██████████| 250/250 [02:11<00:00,  1.90it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 12/12 [00:04<00:00,  2.59it/s]

                   all        368       1860      0.807      0.674        0.7      0.233
EarlyStopping: Training stopped early as no improvement observed in last 30 epochs. Best results observed at epoch 41, best model saved as best.pt.
To update EarlyStopping(patience=30) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



71 epochs completed in 2.706 hours.
Optimizer stripped from /kaggle/working/runs/B0_100ep/weights/last.pt, 6.2MB
Optimizer stripped from /kaggle/working/runs/B0_100ep/weights/best.pt, 6.2MB

Validating /kaggle/working/runs/B0_100ep/weights/best.pt...
Ultralytics 8.3.40 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 168 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 12/12 [00:06<00:00,  1.95it/s]


                   all        368       1860      0.802      0.691      0.761      0.315
          missing_hole         60        300      0.622      0.877      0.715      0.175
            mouse_bite         60        304      0.989      0.602      0.772      0.357
          open_circuit         64        324      0.786      0.694      0.769      0.367
                 short         60        304      0.869      0.707      0.797      0.297
                  spur         60        300       0.56      0.639      0.661      0.282
       spurious_copper         64        328      0.986      0.624      0.855      0.412
Speed: 0.3ms preprocess, 2.6ms inference, 0.0ms loss, 2.6ms postprocess per image
Results saved to /kaggle/working/runs/B0_100ep

Training hoàn tất! Elapsed: 162.7 phút
mAP50    : 0.7614
mAP50-95 : 0.3150
Precision: 0.8019
Recall   : 0.6906


In [27]:
RUN_DIR = WORK_DIR / "runs" / RUN_NAME

# GPU info
gpu_info = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.used",
     "--format=csv,noheader"],
    capture_output=True, text=True
).stdout.strip()

m = results.results_dict
meta = {
    "model"       : "yolov8n.pt",
    "epochs"      : EPOCHS,
    "batch"       : BATCH,
    "imgsz"       : IMGSZ,
    "seed"        : SEED,
    "patience"    : PATIENCE,
    "environment" : "kaggle-gpu",
    "gpu"         : gpu_info,
    "trained_at"  : datetime.now(timezone.utc).isoformat(),
    "elapsed_min" : round(elapsed / 60, 2),
    "train_imgs"  : len(train_paths),
    "val_imgs"    : len(val_paths),
    "map50"       : m.get("metrics/mAP50(B)"),
    "map50_95"    : m.get("metrics/mAP50-95(B)"),
    "precision"   : m.get("metrics/precision(B)"),
    "recall"      : m.get("metrics/recall(B)"),
    "split_source": "project_splits" if splits_loaded and CODE_ROOT else "scan_fallback",
}

with open(RUN_DIR / "run_metadata.json", "w") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

with open(RUN_DIR / "gpu_info.txt", "w") as f:
    f.write(gpu_info + "\n")

print("✅ Metadata saved:")
print(json.dumps(meta, indent=2, ensure_ascii=False))


✅ Metadata saved:
{
  "model": "yolov8n.pt",
  "epochs": 100,
  "batch": 16,
  "imgsz": 640,
  "seed": 41,
  "patience": 30,
  "environment": "kaggle-gpu",
  "gpu": "Tesla T4, 15360 MiB, 687 MiB\nTesla T4, 15360 MiB, 0 MiB",
  "trained_at": "2026-09-24T07:05:54.976710+00:00",
  "elapsed_min": 162.74,
  "train_imgs": 3989,
  "val_imgs": 368,
  "map50": 0.7614489254306948,
  "map50_95": 0.3149536385542759,
  "precision": 0.8019059110749088,
  "recall": 0.6905683822229772,
  "split_source": "project_splits"
}


In [28]:
OUT_ZIP = WORK_DIR / "B0_100ep_weights.zip"
INCLUDE_EXT = {".pt", ".csv", ".yaml", ".json", ".png", ".txt"}

with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in RUN_DIR.rglob("*"):
        if f.is_file() and f.suffix in INCLUDE_EXT:
            zf.write(f, f.relative_to(WORK_DIR))

size_mb = OUT_ZIP.stat().st_size / 1e6
print(f"✅ Packed: B0_100ep_weights.zip  ({size_mb:.1f} MB)")
print("\nNội dung zip:")
with zipfile.ZipFile(OUT_ZIP) as zf:
    for name in sorted(zf.namelist()):
        info = zf.getinfo(name)
        print(f"  {name:55s}  {info.file_size/1024:7.1f} KB")
print("\n👉 Download file này từ Kaggle Output tab")


✅ Packed: B0_100ep_weights.zip  (12.8 MB)

Nội dung zip:
  runs/B0_100ep/F1_curve.png                                 268.6 KB
  runs/B0_100ep/PR_curve.png                                 218.1 KB
  runs/B0_100ep/P_curve.png                                  240.5 KB
  runs/B0_100ep/R_curve.png                                  229.9 KB
  runs/B0_100ep/args.yaml                                      1.5 KB
  runs/B0_100ep/confusion_matrix.png                         149.0 KB
  runs/B0_100ep/confusion_matrix_normalized.png              159.0 KB
  runs/B0_100ep/gpu_info.txt                                   0.1 KB
  runs/B0_100ep/results.csv                                    8.7 KB
  runs/B0_100ep/results.png                                  338.6 KB
  runs/B0_100ep/run_metadata.json                              0.5 KB
  runs/B0_100ep/weights/best.pt                             6091.7 KB
  runs/B0_100ep/weights/last.pt                             6091.7 KB

👉 Download file này từ Kaggle Ou